# Notebook 6: Trend Analysis Dimension Skill

This notebook does two things:

1. **Extraction pipeline** (Groups 2): reads all files in `history_trend_report/`, runs a two-phase LLM extraction to produce `trend_knowledge/dimensions.json`.
2. **`%%writefile` cell** (Group 3): generates `src/deep_research_from_scratch/trend_dimensions.py` — the runtime loader used by Scope and Supervisor nodes.

Re-run the extraction cells any time new files are added to `history_trend_report/`.
Re-run the `%%writefile` cell any time the loader logic changes.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

%load_ext autoreload
%autoreload 2

## Part 1: Extraction Pipeline

In [ ]:
import json
import logging
import os
from datetime import datetime
from pathlib import Path

import pdfplumber
from langchain.chat_models import init_chat_model
from pptx import Presentation
from pydantic import BaseModel

from deep_research_from_scratch.Helper import GenAIToken

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

HISTORY_REPORT_DIR = Path("../history_trend_report")
TREND_KNOWLEDGE_DIR = Path("../trend_knowledge")
TREND_KNOWLEDGE_DIR.mkdir(exist_ok=True)

# Default model for extraction — override via env var EXTRACTION_MODEL
DEFAULT_EXTRACTION_MODEL = os.getenv("EXTRACTION_MODEL", "azure_openai:GPT-54-2026-03-05")
# Truncate each document to this many characters before sending to Phase 1 LLM
MAX_CHARS_PER_DOC = 40_000

### Step 1: Text Extraction Helpers

In [ ]:
def extract_text_from_pdf(filepath: Path) -> dict:
    """Extract text from a PDF file, one chunk per page."""
    pages = []
    try:
        with pdfplumber.open(filepath) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ""
                if text.strip():
                    pages.append(text)
                else:
                    logger.warning("No text on page %d of %s", i + 1, filepath.name)
    except Exception as e:
        logger.error("Failed to open PDF %s: %s", filepath.name, e)
    return {
        "filename": filepath.name,
        "file_type": "pdf",
        "text": "\n\n".join(pages),
        "page_count": len(pages),
    }


def extract_text_from_pptx(filepath: Path) -> dict:
    """Extract text from a PPTX file, one chunk per slide."""
    slides = []
    try:
        prs = Presentation(filepath)
        for i, slide in enumerate(prs.slides):
            texts = []
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text.strip():
                    texts.append(shape.text.strip())
            if texts:
                slides.append("\n".join(texts))
            else:
                logger.warning("No text on slide %d of %s", i + 1, filepath.name)
    except Exception as e:
        logger.error("Failed to open PPTX %s: %s", filepath.name, e)
    return {
        "filename": filepath.name,
        "file_type": "pptx",
        "text": "\n\n".join(slides),
        "slide_count": len(slides),
    }


def extract_document_text(filepath: Path) -> dict:
    """Dispatch text extraction by file extension."""
    suffix = filepath.suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(filepath)
    elif suffix == ".pptx":
        return extract_text_from_pptx(filepath)
    else:
        logger.warning("Unsupported file type: %s", filepath.name)
        return {"filename": filepath.name, "file_type": "unknown", "text": ""}

### Step 2: Pydantic Schemas

In [ ]:
class Dimension(BaseModel):
    """A single analytical dimension used to study trends."""

    name: str
    description: str
    examples: list[str]


class PerDocumentDimensions(BaseModel):
    """Dimensions extracted from a single trend report document."""

    source_doc: str
    dimensions: list[Dimension]


class UnifiedDimensionList(BaseModel):
    """Unified, deduplicated dimension list synthesized across all documents."""

    dimensions: list[Dimension]

### Step 3: LLM Builder

In [ ]:
def build_extraction_model(temperature: float = 0.0):
    """Build the LLM used for dimension extraction."""
    model_id = DEFAULT_EXTRACTION_MODEL
    deployment = model_id.split(":")[-1]
    return init_chat_model(
        model=model_id,
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=deployment,
        api_key=GenAIToken().token(),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        default_headers={
            "project-name": os.getenv("HEADERS_PROJECT_NAME"),
            "userid": os.getenv("HEADERS_USERID"),
        },
        temperature=temperature,
    )

### Step 4: Phase 1 — Per-Document Dimension Extraction

In [ ]:
PHASE1_PROMPT = """You are analyzing a beauty industry trend report.
Your task is NOT to summarize the report's findings or content.
Instead, identify the **analytical dimensions** this report uses to structure
its trend analysis — the lenses or angles through which it examines trends
(e.g., by consumer demographic, by ingredient category, by geographic market,
by distribution channel).

For each dimension you identify:
- `name`: a short label (2-4 words)
- `description`: one sentence explaining what this dimension analyzes
- `examples`: 2-3 concrete examples from the document

Return only dimensions that are genuinely used as a structural lens in this
report. Ignore dimensions mentioned only in passing.

--- DOCUMENT: {filename} ---
{text}"""


def extract_dimensions_from_doc(
    doc: dict,
    model,
) -> PerDocumentDimensions:
    """Phase 1: extract analytical dimensions from a single document."""
    text = doc["text"][:MAX_CHARS_PER_DOC]
    prompt = PHASE1_PROMPT.format(filename=doc["filename"], text=text)
    structured_model = model.with_structured_output(PerDocumentDimensions)
    result = structured_model.invoke(prompt)
    result.source_doc = doc["filename"]
    return result

### Step 5: Phase 2 — Cross-Document Synthesis

In [ ]:
PHASE2_PROMPT = """You have received dimension lists extracted from {n} beauty
industry trend reports. Each list describes the analytical lenses that one
report uses to structure its trend analysis.

Your task: synthesize these into a single, unified, deduplicated list of
**10-20 canonical dimensions**. Merge synonymous or highly overlapping entries
into one (e.g., 'Consumer Age Groups' and 'Generational Segments' should
become 'Consumer Demographics'). Prefer broader, more reusable dimension names.

For each canonical dimension:
- `name`: a short, reusable label (2-4 words)
- `description`: one sentence, generalized across all reports
- `examples`: 2-3 concrete examples that illustrate what this dimension covers

--- PER-DOCUMENT DIMENSION LISTS ---
{all_dimensions}"""


def synthesize_dimensions(
    per_doc_results: list[PerDocumentDimensions],
    model,
) -> UnifiedDimensionList:
    """Phase 2: synthesize per-document dimension lists into one unified list."""
    all_dims_text = ""
    for r in per_doc_results:
        all_dims_text += f"\n### {r.source_doc}\n"
        for d in r.dimensions:
            all_dims_text += f"- **{d.name}**: {d.description}\n"
            all_dims_text += f"  Examples: {', '.join(d.examples)}\n"

    prompt = PHASE2_PROMPT.format(n=len(per_doc_results), all_dimensions=all_dims_text)
    structured_model = model.with_structured_output(UnifiedDimensionList)
    return structured_model.invoke(prompt)

### Step 6: Run the Full Extraction Pipeline

Run this cell to process all documents in `history_trend_report/` and write `trend_knowledge/dimensions.json`.

> **Note:** This cell makes ~13 LLM API calls (12 per-document + 1 synthesis). Typical runtime: 2–5 minutes.

In [ ]:
# 1. Discover all documents
report_files = sorted(HISTORY_REPORT_DIR.glob("*.pdf")) + sorted(HISTORY_REPORT_DIR.glob("*.pptx"))
print(f"Found {len(report_files)} documents:")
for f in report_files:
    print(f"  {f.name}")

# 2. Extract text from each document
print("\nExtracting text...")
documents = [extract_document_text(f) for f in report_files]
docs_with_text = [d for d in documents if d["text"].strip()]
print(f"Text extracted from {len(docs_with_text)}/{len(documents)} documents")

# 3. Phase 1: per-document LLM extraction
model = build_extraction_model()
per_doc_results = []
print("\nPhase 1: extracting dimensions per document...")
for doc in docs_with_text:
    print(f"  [{docs_with_text.index(doc)+1}/{len(docs_with_text)}] {doc['filename']}")
    result = extract_dimensions_from_doc(doc, model)
    per_doc_results.append(result)
    print(f"    → {len(result.dimensions)} dimensions found")

# 4. Phase 2: cross-document synthesis
print("\nPhase 2: synthesizing unified dimension list...")
unified = synthesize_dimensions(per_doc_results, model)
print(f"→ Unified list: {len(unified.dimensions)} canonical dimensions")
for d in unified.dimensions:
    print(f"  - {d.name}: {d.description}")

# 5. Write dimensions.json
output = {
    "extraction_date": datetime.now().strftime("%Y-%m-%d"),
    "source_docs": [d["filename"] for d in docs_with_text],
    "dimensions": [d.model_dump() for d in unified.dimensions],
}
output_path = TREND_KNOWLEDGE_DIR / "dimensions.json"
output_path.write_text(json.dumps(output, indent=2, ensure_ascii=False))
print(f"\nWritten: {output_path} ({len(unified.dimensions)} dimensions)")

## Part 2: Dimension Loader Utility (`%%writefile`)

Run the cell below to regenerate `src/deep_research_from_scratch/trend_dimensions.py`.
This file is the runtime loader used by the Scope and Supervisor nodes — it does **not** make any LLM calls.

In [ ]:
%%writefile ../src/deep_research_from_scratch/trend_dimensions.py

"""Trend analysis dimension loader for prompt enrichment.

Loads pre-extracted analytical dimensions from trend_knowledge/dimensions.json
and formats them for injection into Scope and Supervisor prompts.
Generated by notebooks/6_trend_skill.ipynb — do not edit directly.
"""

import json
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

_CACHE: list[dict] | None = None


def _find_dimensions_file() -> Path | None:
    """Locate trend_knowledge/dimensions.json relative to the repo root."""
    candidates = [
        # Installed package: src/deep_research_from_scratch/ → repo root is 3 levels up
        Path(__file__).resolve().parent.parent.parent / "trend_knowledge" / "dimensions.json",
        # Running from repo root (e.g., notebooks or scripts)
        Path.cwd() / "trend_knowledge" / "dimensions.json",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def load_trend_dimensions() -> list[dict] | None:
    """Load trend analysis dimensions from the pre-generated knowledge file.

    Returns a list of dimension dicts on success, or None if the file is not
    found. Result is cached in memory to avoid repeated disk reads.
    """
    global _CACHE
    if _CACHE is not None:
        return _CACHE

    path = _find_dimensions_file()
    if path is None:
        logger.warning(
            "trend_knowledge/dimensions.json not found. "
            "Run notebooks/6_trend_skill.ipynb to generate it. "
            "Continuing without trend dimension guidance."
        )
        return None

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        _CACHE = data.get("dimensions", [])
        logger.info("Loaded %d trend dimensions from %s", len(_CACHE), path)
        return _CACHE
    except Exception as e:
        logger.warning("Failed to load trend dimensions: %s", e)
        return None


def format_dimensions_for_prompt(dimensions: list[dict] | None) -> str:
    """Format trend dimensions as a compact Markdown list for prompt injection.

    Args:
        dimensions: List of dimension dicts from load_trend_dimensions(), or None.

    Returns:
        Formatted Markdown string, or empty string if dimensions is None/empty.
    """
    if not dimensions:
        return ""

    lines = []
    for d in dimensions:
        name = d.get("name", "")
        description = d.get("description", "")
        examples = d.get("examples", [])
        example_str = ", ".join(examples) if examples else ""
        line = f"- **{name}**: {description}"
        if example_str:
            line += f" (e.g., {example_str})"
        lines.append(line)

    return "\n".join(lines)
